```bash
CUDA_VISIBLE_DEVICES=7 vllm serve Qwen/Qwen2.5-7B-Instruct \
    --host 0.0.0.0 \
    --port 8084 \
    --gpu-memory-utilization 0.85 \
    --enable-prefix-caching \
    --dtype bfloat16 \
    --max_model_len 32000 \
    --trust-remote-code
```

```bash
CUDA_VISIBLE_DEVICES=6 vllm serve Skywork/Skywork-o1-Open-PRM-Qwen-2.5-7B \
    --host 0.0.0.0 \
    --port 8082 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

```bash
CUDA_VISIBLE_DEVICES=5 vllm serve Qwen/Qwen2.5-Math-PRM-7B \
    --host 0.0.0.0 \
    --port 8083 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

In [1]:
from openai import OpenAI

OPENAI_API_KEY = "EMPTY"
OPENAI_API_BASE = "http://localhost:{PORT}/v1"

causal_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE.format(PORT=8084),
)
causal_model = causal_client.models.list().data[0].id

skywork_prm_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE.format(PORT=8082),
)
skywork_prm_model = skywork_prm_client.models.list().data[0].id

qwen_prm_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE.format(PORT=8083),
)
qwen_prm_model = qwen_prm_client.models.list().data[0].id


In [2]:
# def attack(df_sample, 
#     task_text, 
#     experiment_name,
#     use_augment_question_prm=True,
#     prm_clients = [skywork_prm_client, qwen_prm_client],
#     causal_client=causal_client
# ):
#     causal_model = causal_client.models.list().data[0].id
    

#     # Apply the augmentor function to each row in the DataFrame
#     aug_results = augmentor(df_sample, task_text, client=causal_client, model=causal_model)
#     df_aug = pd.DataFrame(aug_results)

#     # Check equivalence
#     equivalence_results = equivalence_check(df_sample, df_aug, client=causal_client, model=causal_model)
#     df_aug["equivalence"] = equivalence_results["equivalence"]
#     df_aug["body_equivalence_results"] = equivalence_results["body_equivalence_results"]

#     # PRM Scorer
#     for prm_client, prm_model in zip(prm_clients, prm_models):
#         rewards = prm_scorer(questions=df["aug_problem"].tolist() if use_augment_question_prm else df["problem"].tolist(),
#                             steps=df["aug_steps"].tolist(), 
#                             client=prm_client, model=prm_model)
#         df[f"{prm_model}--aug_rewards"] = rewards
            
#     # concat the original and augmented DataFrames
#     for key in df_aug.keys():
#         df_sample[key] = df_aug[key]
    
#     # Save the DataFrame to a CSV file
#     df_sample.to_parquet(f"experiments/{experiment_name}.parquet", index=False)
#     return df_sample

In [3]:
import os
import pandas as pd
from constants.prompts_constants import (
    VERBOSE_TASK, CONCISE_TASK, EQ_TO_TEXT_TASK, REPHRASE_TASK, INCORRECT_ASSUMPTION_TASK, CHANGE_NUMBERS_QUESTION_TASK
)
from utils.attack_utils import chatgpt_batch_augmentor, chatgpt_batch_equivalence_checker, prm_scorer

def attack_chatgpt_batch(df,
    task_text, 
    experiment_name,
    prm_clients=None,
    run_augmentor=True,
    run_equivalence_check=True,
    run_prm_scorer=True,
    use_augment_question_prm=False,

):
    """
    Run the attack on the given dataframe using the chatgpt batch API.
    Args:
        df: The dataframe to attack. Should have the following columns:
            - problem: The problem to attack.
            - steps: The steps to attack.
        task_text: The task to use for the attack.
        experiment_name: The name of the experiment.
        prm_clients: The prm clients to use for the attack.
        run_augmentor: Whether to run the augmentor.
        run_equivalence_check: Whether to run the equivalence check.
        run_prm_scorer: Whether to run the prm scorer.
        use_augment_question_prm: Whether to use the augmented question for the prm scorer.
    Returns:
        df: The dataframe with the attack results. 
            For each row, it will have the following columns:
                - problem: The problem to attack.
                - steps: The steps to attack.
                - aug_problem: The augmented question.
                - aug_steps: The augmented steps.
                - equivalence: Whether the augmented question and steps are equivalent to the original question and steps.
                - body_equivalence_results: The body of the equivalence check.
                - {prm_model}--aug_rewards: The rewards from the prm scorer for the augmented question and steps.
    """
    experiment_path = os.path.join("experiments", experiment_name)
    os.makedirs(experiment_path, exist_ok=True)
    if run_augmentor:
        df = chatgpt_batch_augmentor(df, task_text, experiment_path)
    if run_equivalence_check:
        df = chatgpt_batch_equivalence_checker(df, experiment_path)
    if run_prm_scorer:
        prm_models = [prm_client.models.list().data[0].id for prm_client in prm_clients]
        for prm_client, prm_model in zip(prm_clients, prm_models):
            rewards = prm_scorer(questions=df["aug_problem"].tolist() if use_augment_question_prm else df["problem"].tolist(),
                                steps=df["aug_steps"].tolist(), 
                                client=prm_client, model=prm_model)
            df[f"{prm_model}--aug_rewards"] = rewards
    
    df.to_parquet(os.path.join(experiment_path, "attack.parquet"), index=False)
    return df

[2025-05-04 08:44:14,288] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/rishabhtiwari/anaconda3/envs/reasoning2/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/rishabhtiwari/anaconda3/envs/reasoning2/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/rishabhtiwari/anaconda3/envs/reasoning2/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/rishabhtiwari/anaconda3/envs/reasoning2/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/rishabhtiwari/anaconda3/envs/reasoning2/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/rishabhtiwari/anaconda3/envs/reasoning2/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std:

In [8]:
import numpy as np

def attack_chatgpt_batch_shuffled_question(df,
    task_text, 
    experiment_name,
    prm_clients=None,
    run_augmentor=True,
    run_equivalence_check=True,
    run_prm_scorer=True,
    use_augment_question_prm=False,

):
    """
    Run the attack on the given dataframe using the chatgpt batch API.
    Args:
        df: The dataframe to attack. Should have the following columns:
            - prREPHRASE_TASKoblem: The problem to attack.
            - steps: The steps to attack.
        task_text: The task to use for the attack.
        experiment_name: The name of the experiment.
        prm_clients: The prm clients to use for the attack.
        run_augmentor: Whether to run the augmentor.
        run_equivalence_check: Whether to run the equivalence check.
        run_prm_scorer: WhethREPHRASE_TASKer to run the prm scorer.
        use_augment_question_prm: Whether to use the augmented question for the prm scorer.
    Returns:
        df: The dataframe with the attack results. 
            For each row, it will have the following columns:
                - problem: The problem to attack.
                - steps: The steps to attack.
                - aug_problem: The augmented question.
                - aug_steps: The augmented steps.
                - equivalence: Whether the augmented question and steps are equivalent to the original question and steps.
                - body_equivalence_results: The body of the equivalence check.
                - {prm_model}--aug_rewards: The rewards from the prm scorer for the augmented question and steps.
    """
    experiment_path = os.path.join("experiments", experiment_name)
    # Create a shuffled_problem column by randomly shuffling the problems
    
    # Make a copy of the problems
    all_problems = df["problem"].tolist()
    
    # Create a shuffled version of the problems
    shuffled_problems = all_problems.copy()
    np.random.seed(42)  # For reproducibility
    np.random.shuffle(shuffled_problems)
    
    # Assign the shuffled problems to a new column
    df["shuffled_problem"] = shuffled_problems
    os.makedirs(experiment_path, exist_ok=True)
    if run_augmentor:
        df = chatgpt_batch_augmentor(df, task_text, experiment_path)
    if run_equivalence_check:
        df = chatgpt_batch_equivalence_checker(df, experiment_path)
    if run_prm_scorer:
        prm_models = [prm_client.models.list().data[0].id for prm_client in prm_clients]
        for prm_client, prm_model in zip(prm_clients, prm_models):
            rewards = prm_scorer(questions=df["shuffled_problem"].tolist(),
                                steps=df["steps"].tolist(), 
                                client=prm_client, model=prm_model)
            df[f"{prm_model}--aug_rewards"] = rewards
    
    df.to_parquet(os.path.join(experiment_path, "attack.parquet"), index=False)
    return df

In [14]:
import os
import pandas as pd
from constants.prompts_constants import (
    VERBOSE_TASK, CONCISE_TASK, EQ_TO_TEXT_TASK, CHANGE_NUMBERS_TASK, REPHRASE_TASK
)
from utils.attack_utils import chatgpt_batch_augmentor, chatgpt_batch_equivalence_checker, prm_scorer

def attack_chatgpt_batch_remove_question(df,
    task_text, 
    experiment_name,
    prm_clients=None,
    run_augmentor=True,
    run_equivalence_check=True,
    run_prm_scorer=True,
    use_augment_question_prm=False,
    remove_user_chat_template=True,

):
    """
    Run the attack on the given dataframe using the chatgpt batch API.
    Args:
        df: The dataframe to attack. Should have the following columns:
            - prREPHRASE_TASKoblem: The problem to attack.
            - steps: The steps to attack.
        task_text: The task to use for the attack.
        experiment_name: The name of the experiment.
        prm_clients: The prm clients to use for the attack.
        run_augmentor: Whether to run the augmentor.
        run_equivalence_check: Whether to run the equivalence check.
        run_prm_scorer: WhethREPHRASE_TASKer to run the prm scorer.
        use_augment_question_prm: Whether to use the augmented question for the prm scorer.
    Returns:
        df: The dataframe with the attack results. 
            For each row, it will have the following columns:
                - problem: The problem to attack.
                - steps: The steps to attack.
                - aug_problem: The augmented question.
                - aug_steps: The augmented steps.
                - equivalence: Whether the augmented question and steps are equivalent to the original question and steps.
                - body_equivalence_results: The body of the equivalence check.
                - {prm_model}--aug_rewards: The rewards from the prm scorer for the augmented question and steps.
    """
    experiment_path = os.path.join("experiments", experiment_name)
    os.makedirs(experiment_path, exist_ok=True)
    # if run_augmentor:
    #     df = chatgpt_batch_augmentor(df, task_text, experiment_path)
    # if run_equivalence_check:
    #     df = chatgpt_batch_equivalence_checker(df, experiment_path)
    if run_prm_scorer:
        prm_models = [prm_client.models.list().data[0].id for prm_client in prm_clients]
        for prm_client, prm_model in zip(prm_clients, prm_models):
            rewards = prm_scorer(questions=[None]*len(df) if remove_user_chat_template else [""]*len(df),
                                steps=df["steps"].tolist(), 
                                client=prm_client, model=prm_model)
            df[f"{prm_model}--aug_rewards"] = rewards
    
    df.to_parquet(os.path.join(experiment_path, "attack.parquet"), index=False)
    return df

In [15]:
df = pd.read_parquet("data/processbench.parquet")


## TODO: Filter dataset
# sample_size = 100
# df_sample = df.sample(sample_size, random_state=42).reset_index(drop=True)


In [5]:
# df = pd.read_parquet(f"experiments/{experiment_name}/attack.parquet")

In [6]:
# df.head()

In [7]:
# for i in range(len(df)):
#     if df.iloc[i]["aug_problem"]=="":
#         print(i)


In [8]:
# print(df.iloc[2259]['aug_steps'])

In [9]:
for experiment_name, task_text in [("chatgpt_batch_concise", CONCISE_TASK), ("chatgpt_batch_verbose", VERBOSE_TASK), ("chatgpt_batch_rephrase", REPHRASE_TASK)]:
    print(experiment_name)
    df = pd.read_parquet(f"experiments/{experiment_name}/attack.parquet")
    aug_df = attack_chatgpt_batch(df, 
                    task_text=task_text,
                    experiment_name=experiment_name,
                    prm_clients=[skywork_prm_client, qwen_prm_client],
                    run_augmentor=False,
                    run_equivalence_check=False,
                    run_prm_scorer=True,
                    use_augment_question_prm=False
                )

chatgpt_batch_concise


[PRM] Scoring: 100%|██████████| 107/107 [02:59<00:00,  1.68s/it]


chatgpt_batch_verbose


[PRM] Scoring: 100%|██████████| 107/107 [03:40<00:00,  2.06s/it]


chatgpt_batch_rephrase


[PRM] Scoring: 100%|██████████| 107/107 [03:34<00:00,  2.00s/it]


In [ ]:
df = pd.read_parquet("data/processbench.parquet")
for experiment_name, task_text in [("question_removal_no_user_template", '')]:
    print(experiment_name)
    aug_df = attack_chatgpt_batch_remove_question(df, 
                    task_text=task_text,
                    experiment_name=experiment_name,
                    prm_clients=[qwen_prm_client],
                    run_augmentor=False,
                    run_equivalence_check=False,
                    run_prm_scorer=True,
                    use_augment_question_prm=False,
                    remove_user_chat_template=False
                )

question_removal_no_user_template


[PRM] Scoring:  67%|██████▋   | 72/107 [03:21<01:51,  3.20s/it]

In [ ]:
import time
from openai import AzureOpenAI
from constants.private_key import CREDENTIALS_BATCH, ENDPOINT_BATCH, API_VERSION_BATCH

BATCH_ID = "XXXX"

client = AzureOpenAI(
        api_key=CREDENTIALS_BATCH,
        azure_endpoint=ENDPOINT_BATCH,
        api_version=API_VERSION_BATCH
    )
client.batches.cancel(BATCH_ID)
status = "validating"
while status not in ("completed","failed","canceled"):
    status = client.batches.retrieve(BATCH_ID).status
    print(status)
    time.sleep(30)

cancelling
cancelling
cancelled
cancelled


KeyboardInterrupt: 

In [16]:
# import pandas as pd
# aug_df = pd.read_parquet("experiments/chatgpt_batch_verbose/attack.parquet")

In [17]:
# print("Problem:", aug_df["problem"].iloc[3])
# print("Aug Problem:", aug_df["aug_problem"].iloc[3])
# print("--------------------------------")
# print("Steps:", aug_df["steps"].iloc[3])
# print("Aug Steps:", aug_df["aug_steps"].iloc[3])


Problem: A class of 50 students has various hobbies. 10 like to bake, 5 like to play basketball, and the rest like to either play video games or play music. How many like to play video games if the number that like to play music is twice the number that prefer playing basketball?
Aug Problem: There is a class with 50 students who have different hobbies. According to the information available, 10 students have a hobby of baking, 5 students enjoy playing basketball, and the remaining students prefer either playing video games or engaging in musical activities. How many of these students like to play video games, given that the number who enjoy playing music is twice as many as those who prefer playing basketball?
--------------------------------
Steps: ["To find out how many students like to play video games, let's start with the information given: There are 50 students in total. 10 students like to bake. 5 students like to play basketball. The number of students who like to play music i

In [18]:
# print(aug_df.body_equivalence_results.iloc[3])

<step_count>Y</step_count>
  <question_thinking>Both questions are essentially asking how many students like to play video games, given the same initial conditions about baking, basketball, and music preferences.</question_thinking>
  <question>Y</question>
  <step1_thinking>Both Step 1s discuss the initial details provided in the problem and set up the scenario with total students and the breakdown of their hobbies, identifying who likes to bake, play basketball, and the relationship between music and basketball preferences.</step1_thinking>
  <step1>Y</step1>
  <step2_thinking>Both Step 2s calculate the number of students who prefer playing music by multiplying the number of basketball enthusiasts by two.</step2_thinking>
  <step2>Y</step2>
  <step3_thinking>Both Step 3s outline the mathematical operation to find the number of students who like to play video games by subtracting the number of baking and basketball students from the total number.</step3_thinking>
  <step3>Y</step3>
  

In [19]:
# print(aug_df[aug_df.equivalence==False].body_equivalence_results.iloc[0])

<step_count>N</step_count>
  <question_thinking>Both questions ask for how many more pink flamingos are present compared to white flamingos at noon on Sunday.</question_thinking>
  <question>Y</question>
  <step1_thinking>Both Step 1s introduce the initial count of flamingos on Friday morning correctly.</step1_thinking>
  <step1>Y</step1>
  <step2_thinking>Step 2 in both sets correctly computes the one third of flamingos taken, painted white, and sets the counts of pink and white flamingos by end of Saturday.</step2_thinking>
  <step2>Y</step2>
  <step3_thinking>Step 3 in Set A adds 18 pink flamingos to the count from Step 2, reaching 36 pink flamingos and keeps 6 white flamingos. Step 3 in Set B adds 18 to the wrong value, making it 30 pink flamingos instead of 36.</step3_thinking>
  <step3>N</step3>
  <step4_thinking>Both Step 4s carry out the subtraction of white flamingos count from pink, but based on Step 3 error, Set B ends up with different conclusion.</step4_thinking>
  <step4>

In [20]:
# aug_df.equivalence.value_counts()

equivalence
True     3192
False     208
Name: count, dtype: int64